In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense

from IPython.display import display, Markdown

print("TensorFlow:", tf.__version__)
print("Num GPUs:", len(tf.config.list_physical_devices("GPU")))

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "DejaVu Sans"
})

CMAP_FEATURE = "turbo"
CMAP_HEAT = "magma"
CMAP_POSNEG = "coolwarm"

FIG_DIR = "cnn_visualisations"
os.makedirs(FIG_DIR, exist_ok=True)

print("Visualisation directory:", FIG_DIR)

In [ ]:
model = VGG16(weights="imagenet", include_top=True)

print("Model name:", model.name)
print("Total layers:", len(model.layers))
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

In [ ]:
model.summary()

In [ ]:
layer_rows = []

for i, layer in enumerate(model.layers):

    shape = layer.output.shape

    try:
        output_shape = tuple(shape.as_list())
    except AttributeError:
        output_shape = tuple(shape)

    if isinstance(layer, Conv2D):
        layer_type = "Convolution"
        visual_type = "Spatial feature map"
        params = layer.count_params()

    elif isinstance(layer, MaxPooling2D):
        layer_type = "Max Pooling"
        visual_type = "Spatial feature map"
        params = layer.count_params()

    elif isinstance(layer, Dense):
        layer_type = "Dense"
        visual_type = "Vector"
        params = layer.count_params()

    else:
        layer_type = layer.__class__.__name__
        visual_type = "Input / vector / other"
        params = layer.count_params()

    layer_rows.append({
        "Index": i,
        "Layer": layer.name,
        "Type": layer_type,
        "Output Shape": str(output_shape),
        "Parameters": f"{params:,}",
        "How to visualize": visual_type
    })

layer_table = pd.DataFrame(layer_rows)

display(layer_table)

print("\n" + "=" * 60)
print("CNN ARCHITECTURE SUMMARY")
print("=" * 60)

print(
    "\nConvolution layers:",
    sum(isinstance(l, Conv2D) for l in model.layers)
)

print(
    "Pooling layers:",
    sum(isinstance(l, MaxPooling2D) for l in model.layers)
)

print(
    "Dense layers:",
    sum(isinstance(l, Dense) for l in model.layers)
)

print(
    "Total layers:",
    len(model.layers)
)


In [ ]:
conv_layers = [l for l in model.layers if isinstance(l, Conv2D)]
pool_layers = [l for l in model.layers if isinstance(l, MaxPooling2D)]
dense_layers = [l for l in model.layers if isinstance(l, Dense)]

spatial_layers = conv_layers + pool_layers
spatial_layers = sorted(spatial_layers, key=lambda layer: model.layers.index(layer))

print("Convolution layers:")
print([l.name for l in conv_layers])

print("\nPooling layers:")
print([l.name for l in pool_layers])

print("\nDense layers:")
print([l.name for l in dense_layers])

In [ ]:
IMAGE_PATH = "/content/ElonMusk.jpg"

if not os.path.exists(IMAGE_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            IMAGE_PATH = next(iter(uploaded.keys()))
    except Exception:
        IMAGE_PATH = input("Enter image path: ").strip()

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(
        f"Image not found: {IMAGE_PATH}\n"
        "Upload an image or update IMAGE_PATH."
    )

print("Using image:", IMAGE_PATH)

original_pil = load_img(IMAGE_PATH)
original_rgb = np.array(original_pil.convert("RGB"))

model_input_pil = load_img(
    IMAGE_PATH,
    target_size=(224, 224)
)

image_array = img_to_array(model_input_pil)
image_batch = np.expand_dims(image_array, axis=0)
image_preprocessed = preprocess_input(image_batch.copy())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(original_rgb)
axes[0].set_title("Original Image", fontsize=18, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(np.clip(image_array / 255.0, 0, 1))
axes[1].set_title("224 × 224 Image Fed to VGG16", fontsize=18, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
first_conv = conv_layers[0]

filters, biases = first_conv.get_weights()

print("Layer:", first_conv.name)
print("Filter tensor shape:", filters.shape)
print("Bias shape:", biases.shape)

def normalize_filter(f):
    f_min = f.min()
    f_max = f.max()
    return (f - f_min) / (f_max - f_min + 1e-8)

n_show = min(24, filters.shape[-1])

fig, axes = plt.subplots(6, 12, figsize=(18, 10))
axes = np.asarray(axes)

for k in range(n_show):
    r = (k * 3) // 12
    c = (k * 3) % 12

    f = normalize_filter(filters[:, :, :, k])

    for ch in range(3):
        ax = axes[r, c + ch]
        ax.imshow(f[:, :, ch], cmap=CMAP_FEATURE)
        ax.set_title(f"F{k+1} · RGB{ch+1}", fontsize=7)
        ax.axis("off")

for ax in axes.ravel():
    if not ax.has_data():
        ax.axis("off")

fig.suptitle(
    f"First-Layer Learned Filters — {first_conv.name}",
    fontsize=20,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
spatial_outputs = [layer.output for layer in spatial_layers]

activation_model = Model(
    inputs=model.inputs,
    outputs=spatial_outputs
)

spatial_activations = activation_model.predict(
    image_preprocessed,
    verbose=0
)

print("Collected activations:", len(spatial_activations))
for layer, activation in zip(spatial_layers, spatial_activations):
    print(f"{layer.name:18s} -> {activation.shape}")

In [ ]:
activation_rows = []

for layer, activation in zip(spatial_layers, spatial_activations):
    a = activation[0]

    activation_rows.append({
        "Index": model.layers.index(layer),
        "Layer": layer.name,
        "Type": layer.__class__.__name__,
        "Height": a.shape[0],
        "Width": a.shape[1],
        "Channels": a.shape[2],
        "Mean": float(np.mean(a)),
        "Std": float(np.std(a)),
        "Min": float(np.min(a)),
        "Max": float(np.max(a)),
        "Active %": float(np.mean(a > 0) * 100)
    })

activation_table = pd.DataFrame(activation_rows)

display(
    activation_table.style
    .format({
        "Mean": "{:.3f}",
        "Std": "{:.3f}",
        "Min": "{:.3f}",
        "Max": "{:.3f}",
        "Active %": "{:.2f}%"
    })
    .background_gradient(subset=["Mean", "Std", "Active %"], cmap="viridis")
)

In [ ]:
def mean_activation_map(activation):
    a = activation[0]
    return np.mean(np.abs(a), axis=-1)

n_layers = len(spatial_layers)
n_cols = 5
n_rows = int(np.ceil(n_layers / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(20, 4.2 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, layer, activation in zip(axes, spatial_layers, spatial_activations):
    heat = mean_activation_map(activation)

    im = ax.imshow(heat, cmap=CMAP_HEAT)
    ax.set_title(
        f"{model.layers.index(layer)} · {layer.name}\n"
        f"{heat.shape[0]}×{heat.shape[1]}",
        fontweight="bold"
    )
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

for ax in axes[n_layers:]:
    ax.axis("off")

fig.suptitle(
    "How the Image Changes Through Every Spatial Layer",
    fontsize=23,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
def strongest_channel(activation):
    a = activation[0]
    scores = np.mean(np.abs(a), axis=(0, 1))
    return int(np.argmax(scores)), float(np.max(scores))

selected = []

for layer, activation in zip(spatial_layers, spatial_activations):
    channel_idx, score = strongest_channel(activation)
    selected.append((layer, activation, channel_idx, score))

n_cols = 4
n_rows = int(np.ceil(len(selected) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(17, 4.2 * n_rows)
)
axes = np.array(axes).reshape(-1)

for ax, (layer, activation, ch, score) in zip(axes, selected):
    fmap = activation[0, :, :, ch]
    vmax = np.percentile(fmap, 99.5) if np.any(fmap > 0) else 1

    ax.imshow(fmap, cmap=CMAP_FEATURE, vmin=0, vmax=vmax)
    ax.set_title(
        f"{layer.name}\nStrongest channel: {ch}  |  score: {score:.2f}",
        fontsize=10,
        fontweight="bold"
    )
    ax.axis("off")

for ax in axes[len(selected):]:
    ax.axis("off")

fig.suptitle(
    "Representative Strongest Feature — Layer by Layer",
    fontsize=22,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
def ranked_channels(activation, n_maps):
    a = activation[0]
    scores = np.mean(np.abs(a), axis=(0, 1))
    return np.argsort(scores)[::-1][:n_maps]

def show_feature_gallery(
    layer,
    activation,
    n_maps=25,
    cmap=CMAP_FEATURE,
    title_prefix=""
):
    a = activation[0]
    channels = ranked_channels(activation, min(n_maps, a.shape[-1]))

    n_cols = 5
    n_rows = int(np.ceil(len(channels) / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(17, 3.5 * n_rows)
    )
    axes = np.array(axes).reshape(-1)

    for ax, ch in zip(axes, channels):
        fmap = a[:, :, ch]

        if np.max(fmap) > 0:
            vmax = np.percentile(fmap, 99.5)
        else:
            vmax = 1

        im = ax.imshow(
            fmap,
            cmap=cmap,
            vmin=0,
            vmax=max(vmax, 1e-8)
        )
        ax.set_title(
            f"Channel {ch}",
            fontsize=9,
            fontweight="bold"
        )
        ax.axis("off")

    for ax in axes[len(channels):]:
        ax.axis("off")

    fig.suptitle(
        f"{title_prefix}{layer.name} — {a.shape[0]}×{a.shape[1]}×{a.shape[2]}",
        fontsize=19,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)

for layer in conv_layers:
    activation = spatial_activations[spatial_layers.index(layer)]
    show_feature_gallery(
        layer,
        activation,
        n_maps=25,
        title_prefix="Convolution Feature Maps · "
    )

In [ ]:
for layer in pool_layers:
    activation = spatial_activations[spatial_layers.index(layer)]
    show_feature_gallery(
        layer,
        activation,
        n_maps=25,
        cmap="plasma",
        title_prefix="Pooling Feature Maps · "
    )

In [ ]:
def resize_heatmap(heat, size=(224, 224)):
    heat = np.asarray(heat, dtype=np.float32)
    heat_min, heat_max = heat.min(), heat.max()
    heat = (heat - heat_min) / (heat_max - heat_min + 1e-8)

    heat_img = Image.fromarray(np.uint8(heat * 255))
    heat_img = heat_img.resize(size, Image.Resampling.BILINEAR)

    return np.asarray(heat_img) / 255.0

base_img = np.clip(image_array / 255.0, 0, 1)

n_cols = 4
n_rows = int(np.ceil(len(spatial_layers) / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 4.2 * n_rows)
)
axes = np.array(axes).reshape(-1)

for ax, layer, activation in zip(
    axes,
    spatial_layers,
    spatial_activations
):
    heat = resize_heatmap(mean_activation_map(activation))

    ax.imshow(base_img)
    ax.imshow(
        heat,
        cmap=CMAP_HEAT,
        alpha=0.50,
        vmin=0,
        vmax=1
    )

    ax.set_title(
        f"{layer.name}\n"
        f"{activation.shape[1]}×{activation.shape[2]} spatial",
        fontsize=10,
        fontweight="bold"
    )
    ax.axis("off")

for ax in axes[len(spatial_layers):]:
    ax.axis("off")

fig.suptitle(
    "Where Is the Network Responding? — Every Spatial Layer",
    fontsize=22,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
rf = 1
jump = 1

rf_rows = []

for layer in model.layers:
    if isinstance(layer, Conv2D):
        kernel = layer.kernel_size[0]
        stride = layer.strides[0]
        rf = rf + (kernel - 1) * jump
        jump = jump * stride

        spatial = layer.output.shape
        rf_rows.append({
            "Layer": layer.name,
            "Type": "Conv",
            "Spatial": f"{spatial[1]}×{spatial[2]}",
            "Channels": int(spatial[3]),
            "Jump": jump,
            "Receptive Field": rf
        })

    elif isinstance(layer, MaxPooling2D):
        kernel = layer.pool_size[0]
        stride = layer.strides[0]
        rf = rf + (kernel - 1) * jump
        jump = jump * stride

        spatial = layer.output.shape
        rf_rows.append({
            "Layer": layer.name,
            "Type": "Pool",
            "Spatial": f"{spatial[1]}×{spatial[2]}",
            "Channels": int(spatial[3]),
            "Jump": jump,
            "Receptive Field": rf
        })

rf_table = pd.DataFrame(rf_rows)
display(rf_table)

fig, ax1 = plt.subplots(figsize=(15, 6))

x = np.arange(len(rf_table))

line1 = ax1.plot(
    x,
    rf_table["Receptive Field"],
    marker="o",
    linewidth=3,
    label="Receptive field"
)

ax1.set_xlabel("Depth")
ax1.set_ylabel("Theoretical receptive field (input pixels)")
ax1.set_xticks(x)
ax1.set_xticklabels(rf_table["Layer"], rotation=45, ha="right")

ax2 = ax1.twinx()
line2 = ax2.plot(
    x,
    rf_table["Channels"],
    marker="s",
    linewidth=3,
    label="Channels"
)
ax2.set_ylabel("Number of channels")

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")

plt.title(
    "VGG16 Depth: Larger Receptive Fields + More Feature Channels",
    fontsize=18,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
stats_plot = activation_table.copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

x = np.arange(len(stats_plot))

axes[0, 0].plot(x, stats_plot["Mean"], marker="o", linewidth=2.5)
axes[0, 0].set_title("Mean Activation", fontweight="bold")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(stats_plot["Layer"], rotation=45, ha="right")

axes[0, 1].plot(x, stats_plot["Std"], marker="o", linewidth=2.5)
axes[0, 1].set_title("Activation Standard Deviation", fontweight="bold")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(stats_plot["Layer"], rotation=45, ha="right")

axes[1, 0].plot(x, stats_plot["Max"], marker="o", linewidth=2.5)
axes[1, 0].set_title("Maximum Activation", fontweight="bold")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(stats_plot["Layer"], rotation=45, ha="right")

axes[1, 1].plot(x, stats_plot["Active %"], marker="o", linewidth=2.5)
axes[1, 1].set_title("Percentage of Positive Activations", fontweight="bold")
axes[1, 1].set_ylabel("% active")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(stats_plot["Layer"], rotation=45, ha="right")

fig.suptitle(
    "Activation Statistics — The Numerical Story Behind the Pictures",
    fontsize=21,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
# Build an activation model for the non-spatial layers we want to inspect.
inspection_layers = [
    model.layers[-4],  # flatten
    model.layers[-3],  # fc1
    model.layers[-2],  # fc2
    model.layers[-1],  # predictions
]

inspection_model = Model(
    inputs=model.inputs,
    outputs=[layer.output for layer in inspection_layers]
)

inspection_activations = inspection_model.predict(
    image_preprocessed,
    verbose=0
)

for layer, activation in zip(inspection_layers, inspection_activations):
    print(f"{layer.name:18s} -> {activation.shape}")

In [ ]:
flatten_activation = inspection_activations[0][0]
fc1_activation = inspection_activations[1][0]
fc2_activation = inspection_activations[2][0]
prediction_vector = inspection_activations[3][0]

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].hist(
    flatten_activation,
    bins=80,
    alpha=0.85
)
axes[0].set_title(
    f"Flatten Activation Distribution\nn={flatten_activation.size:,}",
    fontweight="bold"
)
axes[0].set_xlabel("Activation")
axes[0].set_ylabel("Frequency")

axes[1].hist(
    fc1_activation,
    bins=80,
    alpha=0.85
)
axes[1].set_title(
    f"{model.layers[-3].name} Distribution\nn={fc1_activation.size:,}",
    fontweight="bold"
)
axes[1].set_xlabel("Activation")

axes[2].hist(
    fc2_activation,
    bins=80,
    alpha=0.85
)
axes[2].set_title(
    f"{model.layers[-2].name} Distribution\nn={fc2_activation.size:,}",
    fontweight="bold"
)
axes[2].set_xlabel("Activation")

fig.suptitle(
    "From Spatial Features to High-Dimensional Feature Vectors",
    fontsize=20,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
def top_units(values, n=20):
    idx = np.argsort(values)[-n:][::-1]
    return idx, values[idx]

for layer_name, values in [
    (model.layers[-3].name, fc1_activation),
    (model.layers[-2].name, fc2_activation)
]:
    idx, vals = top_units(values, 20)

    fig, ax = plt.subplots(figsize=(13, 6))

    ax.bar(
        [str(i) for i in idx],
        vals
    )

    ax.set_title(
        f"Top 20 Activated Units — {layer_name}",
        fontsize=18,
        fontweight="bold"
    )
    ax.set_xlabel("Unit index")
    ax.set_ylabel("Activation")

    plt.tight_layout()
    plt.show()

In [ ]:
predictions = decode_predictions(
    inspection_activations[-1],
    top=10
)[0]

pred_df = pd.DataFrame(
    predictions,
    columns=["Class ID", "Class Name", "Probability"]
)

display(
    pred_df.style
    .format({"Probability": "{:.2%}"})
    .background_gradient(
        subset=["Probability"],
        cmap="viridis"
    )
)

fig, ax = plt.subplots(figsize=(13, 7))

labels = pred_df["Class Name"]
values = pred_df["Probability"] * 100

ax.barh(
    labels[::-1],
    values[::-1]
)

ax.set_xlabel("Predicted probability (%)")
ax.set_title(
    "Top 10 ImageNet Predictions",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:

# Pick visually meaningful checkpoints while keeping the full all-layer analysis above.
checkpoint_indices = [
    0,
    min(2, len(spatial_layers)-1),
    min(6, len(spatial_layers)-1),
    min(12, len(spatial_layers)-1),
    len(spatial_layers)-1
]

fig, axes = plt.subplots(1, 5, figsize=(23, 5))

axes[0].imshow(base_img)
axes[0].set_title("Input Image", fontsize=14, fontweight="bold")
axes[0].axis("off")

for ax, idx in zip(axes[1:], checkpoint_indices[1:]):
    layer = spatial_layers[idx]
    activation = spatial_activations[idx]
    heat = resize_heatmap(mean_activation_map(activation))

    ax.imshow(base_img)
    ax.imshow(
        heat,
        cmap=CMAP_HEAT,
        alpha=0.55,
        vmin=0,
        vmax=1
    )

    ax.set_title(
        f"{layer.name}\n"
        f"{activation.shape[1]}×{activation.shape[2]}",
        fontsize=12,
        fontweight="bold"
    )
    ax.axis("off")

fig.suptitle(
    "From Raw Pixels to High-Level Visual Representation",
    fontsize=22,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
def run_cnn_visualisation(image_path, model=model):
    pil = load_img(image_path, target_size=(224, 224))
    arr = img_to_array(pil)
    batch = preprocess_input(np.expand_dims(arr.copy(), axis=0))

    activations = activation_model.predict(batch, verbose=0)

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(18, 9)
    )

    axes = axes.ravel()

    rgb = np.clip(arr / 255.0, 0, 1)
    axes[0].imshow(rgb)
    axes[0].set_title("Input", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    checkpoint_ids = np.linspace(
        0,
        len(spatial_layers) - 1,
        7,
        dtype=int
    )

    for ax, idx in zip(axes[1:], checkpoint_ids):
        layer = spatial_layers[idx]
        heat = resize_heatmap(
            mean_activation_map(activations[idx])
        )

        ax.imshow(rgb)
        ax.imshow(
            heat,
            cmap=CMAP_HEAT,
            alpha=0.55,
            vmin=0,
            vmax=1
        )
        ax.set_title(
            layer.name,
            fontsize=11,
            fontweight="bold"
        )
        ax.axis("off")

    plt.suptitle(
        f"CNN Vision Across Depth — {os.path.basename(image_path)}",
        fontsize=20,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

    pred = model.predict(batch, verbose=0)
    decoded = decode_predictions(pred, top=5)[0]

    return pd.DataFrame(
        decoded,
        columns=["Class ID", "Class Name", "Probability"]
    )

# Example:
# new_predictions = run_cnn_visualisation("/content/another_image.jpg")
# display(new_predictions)

In [ ]:
layer_table.to_csv(
    os.path.join(FIG_DIR, "vgg16_layer_inventory.csv"),
    index=False
)

activation_table.to_csv(
    os.path.join(FIG_DIR, "vgg16_activation_statistics.csv"),
    index=False
)

rf_table.to_csv(
    os.path.join(FIG_DIR, "vgg16_receptive_field_table.csv"),
    index=False
)

pred_df.to_csv(
    os.path.join(FIG_DIR, "vgg16_top_predictions.csv"),
    index=False
)

print(f"Saved analysis tables to: {FIG_DIR}/")